# Geneformer in-silico perturbation: TXNDC15 / SYVN1 / MARCHF6 in liver

**Goal.** Take three ERAD-related CRISPR-screen hits (TXNDC15 = your novel hit; SYVN1/HRD1 and MARCHF6 = known E3 ligases). For each, ask Geneformer: *if this gene were knocked out in a hepatocyte, which other genes would shift most in their rank-ordered expression?* Then intersect the three lists to find the broader ERAD network around your hits.

**What Geneformer is.** A transformer trained on ~30M single-cell transcriptomes. Inputs are per-cell rank-ordered gene tokens. "Knockout" = remove that gene's token from the cell's input, re-embed, compare to the unperturbed embedding. The genes whose embedding *position* shifts most are the ones the model thinks are functionally downstream.

**Honesty note.** Geneformer's in-silico KO is a learned embedding shift, not a simulation. It's a hypothesis generator. Use it to rank candidates for wet-lab follow-up, not to draw conclusions.

**Runtime.** ~30–60 min on a Colab T4 with 5–10k cells. Free tier is enough.

## 0. Setup

In [ ]:
# Mount Drive for persistent outputs (optional but recommended)
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = "/content/drive/MyDrive/benchmate_geneformer"
os.makedirs(OUT, exist_ok=True)
print("Outputs ->", OUT)

In [ ]:
# Pin transformers to a version Geneformer is compatible with.
# Recent Colab ships transformers>=4.46 which removed SpecialTokensMixin
# (Geneformer still imports it). Pinning fixes ImportError on tokenizer.
# RESTART RUNTIME after this cell finishes, then continue from Cell 1.
!pip install -q "numpy<2" "transformers==4.40.0" "tokenizers<0.20" \
                "accelerate<1.0" "datasets<3.0" "pandas==2.2.2"
!pip install -q --no-deps git+https://huggingface.co/ctheodoris/Geneformer
!pip install -q cellxgene-census scanpy gseapy loompy
print("\nInstall done. >>> Runtime → Restart session <<<")
print("After restart, re-run from Cell 1 onward.")

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU — InSilicoPerturber will be very slow.")
    print("In Colab: Runtime → Change runtime type → T4 GPU")

## 1. Pull hepatocyte data from CELLxGENE Census

CELLxGENE Census is a versioned snapshot of all human + mouse scRNA-seq deposited to CELLxGENE Discover. Filtering server-side keeps us under Colab's RAM budget.

We're grabbing **healthy human hepatocytes** as our reference cell state. Future runs: swap in disease (e.g. NASH, fibrotic liver) for a disease-vs-healthy contrast.

In [ ]:
import cellxgene_census

CENSUS_VERSION = "2024-07-01"  # pinned for reproducibility
N_CELLS = 8000                  # tune for your compute budget

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        obs_value_filter=(
            "tissue_general == 'liver' "
            "and cell_type == 'hepatocyte' "
            "and disease == 'normal' "
            "and is_primary_data == True"
        ),
        column_names={
            "obs": ["cell_type", "tissue", "disease", "assay", "donor_id"],
            "var": ["feature_id", "feature_name"],
        },
    )

# Down-sample for speed
import numpy as np
if adata.n_obs > N_CELLS:
    idx = np.random.default_rng(0).choice(adata.n_obs, N_CELLS, replace=False)
    adata = adata[idx].copy()

print(f"Cells: {adata.n_obs}  Genes: {adata.n_vars}")
print(adata.obs["assay"].value_counts().head())

In [ ]:
# Geneformer requires:
#   - var.ensembl_id  (gene IDs)
#   - obs.n_counts    (total counts per cell)
# CELLxGENE stores ensembl IDs in var.feature_id, so we alias.
adata.var["ensembl_id"] = adata.var["feature_id"]
adata.obs["n_counts"] = adata.X.sum(axis=1).A1 if hasattr(adata.X, 'A1') else adata.X.sum(axis=1)

# Save as h5ad — Geneformer's tokenizer reads from disk
RAW_PATH = f"{OUT}/hepatocyte_raw.h5ad"
adata.write_h5ad(RAW_PATH)
print("saved:", RAW_PATH)

## 2. Tokenize for Geneformer

Geneformer's tokenizer converts each cell into a rank-ordered list of (Ensembl-id) gene tokens. The model learns embeddings over these token sequences. This step writes a HuggingFace `Dataset` to disk.

In [ ]:
import os, shutil
from geneformer import TranscriptomeTokenizer

# Tokenizer expects a directory of .loom or .h5ad files
TOK_IN = "/content/tok_in"
TOK_OUT = "/content/tok_out"
for d in (TOK_IN, TOK_OUT):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

shutil.copy(RAW_PATH, f"{TOK_IN}/hep.h5ad")

tk = TranscriptomeTokenizer(
    custom_attr_name_dict={"cell_type": "cell_type", "donor_id": "donor_id"},
    nproc=2,
)
tk.tokenize_data(TOK_IN, TOK_OUT, "hep", file_format="h5ad")
print("tokenized ->", os.listdir(TOK_OUT))

## 3. In-silico perturbation

For each gene (TXNDC15, SYVN1, MARCHF6) we run `InSilicoPerturber` with `perturb_type="delete"`. The output is a `.pickle` per gene containing, for every cell, how much each *other* gene's embedding shifted under the perturbation.

Gene IDs below are pinned to Ensembl. Verify on ensembl.org if you swap species or release.

In [ ]:
TARGETS = {
    "TXNDC15":  "ENSG00000113838",
    "SYVN1":    "ENSG00000162298",
    "MARCHF6":  "ENSG00000145495",
}

TOK_DATASET = f"{TOK_OUT}/hep.dataset"
PERTURB_OUT = f"{OUT}/perturbations"
os.makedirs(PERTURB_OUT, exist_ok=True)

In [ ]:
# Download the Geneformer model checkpoint locally — InSilicoPerturber
# needs a real path, not a HF repo ID.
from huggingface_hub import snapshot_download
GENEFORMER_REPO = snapshot_download(repo_id="ctheodoris/Geneformer")
print("Available model subdirs:")
for entry in sorted(os.listdir(GENEFORMER_REPO)):
    sub = os.path.join(GENEFORMER_REPO, entry)
    if os.path.isdir(sub) and "config.json" in os.listdir(sub):
        print("  ", entry)
MODEL_PATH = f"{GENEFORMER_REPO}/Geneformer-V2-104M"
assert os.path.isdir(MODEL_PATH), f"Not found: {MODEL_PATH}"
print("\nUsing model:", MODEL_PATH)

In [ ]:
from geneformer import InSilicoPerturber
import torch, gc

def run_perturbation(gene_symbol, ensembl_id):
    out_dir = f"{PERTURB_OUT}/{gene_symbol}"
    os.makedirs(out_dir, exist_ok=True)
    isp = InSilicoPerturber(
        perturb_type="delete",
        genes_to_perturb=[ensembl_id],
        model_type="Pretrained",
        num_classes=0,
        emb_mode="cls_and_gene",   # V2 uses <cls>
        cell_emb_style="mean_pool",
        filter_data=None,
        cell_states_to_model=None,
        state_embs_dict=None,
        max_ncells=2000,
        emb_layer=0,
        forward_batch_size=4,       # T4 16GB: 16 OOMs with i4096 seqs
        nproc=2,
    )
    isp.perturb_data(
        model_directory=MODEL_PATH,
        input_data_file=TOK_DATASET,
        output_directory=out_dir,
        output_prefix=gene_symbol,
    )
    del isp; gc.collect(); torch.cuda.empty_cache()
    print(f"  -> wrote {gene_symbol} perturbation results to {out_dir}")

for sym, eid in TARGETS.items():
    print(f"\nPerturbing {sym} ({eid})...")
    run_perturbation(sym, eid)

## 4. Statistics: which genes shifted most under each KO?

In [ ]:
from geneformer import InSilicoPerturberStats

def run_stats(gene_symbol):
    in_dir = f"{PERTURB_OUT}/{gene_symbol}"
    stats = InSilicoPerturberStats(
        mode="aggregate_gene_shifts"  # per-gene shifts, no cell-state needed,
        genes_perturbed="all",
        combos=0,
        anchor_gene=None,
        cell_states_to_model=None,
    )
    stats.get_stats(
        input_data_directory=in_dir,
        null_dist_data_directory=None,
        output_directory=in_dir,
        output_prefix=f"{gene_symbol}_stats",
    )
    return f"{in_dir}/{gene_symbol}_stats.csv"

stats_files = {sym: run_stats(sym) for sym in TARGETS}
print("\nStats files:")
for k, v in stats_files.items():
    print(" ", k, "->", v)

## 5. Cross-intersect: the ERAD network around your hits

Genes that shift significantly under **all three** perturbations are the most likely shared downstream nodes — i.e. the broader ERAD/protein-quality-control network the three CRISPR hits live in.

In [ ]:
import pandas as pd

def load_stats(path, top_n=200):
    df = pd.read_csv(path)
    # The stats CSV contains per-gene shift magnitudes; column names depend on
    # the Geneformer release. Adjust the sort column if your version differs.
    score_col = next((c for c in df.columns if "shift" in c.lower() or "score" in c.lower()), df.columns[-1])
    df = df.sort_values(score_col, ascending=False)
    return df.head(top_n), score_col

tops = {}
for sym, path in stats_files.items():
    df, col = load_stats(path)
    tops[sym] = df
    print(f"{sym}: top gene = {df.iloc[0].to_dict()}  (sorted by {col})")

# Intersect top-200 hits across all three perturbations
id_col = "Gene" if "Gene" in tops["TXNDC15"].columns else tops["TXNDC15"].columns[0]
shared = set(tops["TXNDC15"][id_col])
for sym in ("SYVN1", "MARCHF6"):
    shared &= set(tops[sym][id_col])

print(f"\nGenes shifted in all three perturbations: {len(shared)}")
print(sorted(shared)[:30])

In [ ]:
# Save the shared-network shortlist
shared_path = f"{OUT}/erad_shared_network.csv"
pd.DataFrame({"gene": sorted(shared)}).to_csv(shared_path, index=False)
print("saved:", shared_path)

## 6. Pathway enrichment on the shared hits

Take the shared gene list, push it through Enrichr to see which biological processes it lights up. If ERAD / unfolded protein response / proteasome appear at the top, that's confirmation Geneformer recovered known biology around your CRISPR hits.

In [ ]:
import gseapy as gp

# Geneformer outputs Ensembl IDs; Enrichr wants gene symbols.
# Use the AnnData mapping we already have.
ens_to_sym = dict(zip(adata.var["feature_id"], adata.var["feature_name"]))
shared_symbols = [ens_to_sym.get(g, g) for g in shared if g in ens_to_sym]
print(f"Mapped {len(shared_symbols)}/{len(shared)} genes to symbols")

if shared_symbols:
    enr = gp.enrichr(
        gene_list=shared_symbols,
        gene_sets=["GO_Biological_Process_2023", "Reactome_2022", "KEGG_2021_Human"],
        organism="human",
        outdir=f"{OUT}/enrichment",
    )
    print(enr.results.sort_values("Adjusted P-value").head(15)[["Gene_set", "Term", "Adjusted P-value", "Genes"]])

## Next steps

- **Re-run on disease tissue.** Swap the CELLxGENE filter from `disease == 'normal'` to NASH / fibrotic liver / HCC and see whether the shared network shifts. That's a real biological question.
- **Score every gene in the genome, not just three.** Replace `TARGETS` with a panel of E3 ligases or ER-resident genes — gives you a Geneformer-derived "ERAD master regulator" ranking.
- **Plug into Benchmate.** Add a `geneformer_perturb(gene_symbol)` tool to `co_scientist/tools.py`; the Generation agent can then call it when the research goal mentions targets or cell states.
- **Swap to BioNeMo for production.** The model used here is identical to NVIDIA's bundled Geneformer; the swap is mostly a Docker setup, not a code change.